# RAG Retrieval Evaluation: Encoder Comparison
## MiniLM vs PubMedBERT vs BioLORD on a Parkinson's Cell Therapy corpus

This notebook evaluates whether domain-specific biomedical encoders improve semantic retrieval quality compared to a general-purpose encoder, using a manually annotated query set.

**What is being evaluated:** the retrieval step only — ChromaDB's ability to return relevant abstracts for a given query. The LLM generation layer is not involved.

**Metric:** Precision@5 — proportion of retrieved abstracts (top 5) judged relevant by a domain expert.

**Encoders compared:**
- `all-MiniLM-L6-v2` — general-purpose, fast (baseline)
- `microsoft/BiomedNLP-PubMedBERT-base-uncased` — trained on PubMed abstracts
- `FremyCompany/BioLORD-2023` — trained on biomedical concepts, strong on semantic similarity

**Corpus:** 100 PubMed abstracts on "Stem cell therapy Parkinson's disease", fixed across all encoders.

In [ ]:
!pip install requests sentence-transformers chromadb anthropic python-dotenv -q

In [ ]:
import requests
import time
import re
import os
import json
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

print(f"NCBI key loaded: {NCBI_API_KEY is not None}")

## 1. Query Set

14 queries across three categories. Relevance annotation rules:
- **Relevant (1):** abstract directly addresses the main subject of the query
- **Not relevant (0):** abstract only mentions the topic in passing, or is unrelated
- For comparative questions: abstract must address the comparison or specific outcome, not just mention one therapy

In [ ]:
QUERIES = {
    # Specific
    "Q01": "Can MSCs restore motor function of preclinical Parkinsonian models?",
    "Q02": "What are the expected mechanisms of action of dopaminergic progenitors once transplanted in a Parkinsonian brain?",
    "Q03": "Can allogeneic iPSC-derived dopaminergic progenitors escape the immune system of the host?",
    "Q04": "What is the specific neuronal subtype that should be replaced in a Parkinsonian patient?",
    "Q05": "What is the optimal dose of dopaminergic progenitors or dopaminergic neurons to rescue motor symptoms?",
    "Q06": "Can dopaminergic cell therapy relieve non-motor symptoms in Parkinson patients?",
    # Clinical
    "Q07": "What is the optimal disease stage of a Parkinsonian patient to be included in a cell therapy clinical trial?",
    "Q08": "What are the main criteria for patient stratification in a Parkinson cell therapy trial?",
    "Q09": "What is the best immunosuppressive protocol for allogeneic cell therapy for a Parkinson patient?",
    "Q10": "Is cell therapy more effective than DBS for treating Parkinson's disease?",
    "Q11": "Are there reported cases of teratoma formation in iPSC-derived cell therapy for Parkinson disease?",
    # Out-of-scope
    "Q12": "What biomarkers are indicative of liver failure?",
    "Q13": "Are there any therapies able to cure or alleviate Huntington's disease?",
    "Q14": "What is the risk of injecting hyaluronic acid subcutaneously?",
}

print(f"{len(QUERIES)} queries loaded")
for qid, q in QUERIES.items():
    print(f"  {qid}: {q}")

## 2. Corpus

Fetch 100 abstracts from PubMed on the fixed topic. This corpus is identical for all encoder comparisons.

In [ ]:
def fetch_pubmed_abstracts_batched(query, max_results=100, batch_size=50):
    resp = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params={"db":"pubmed","term":query,"retmax":max_results,"retmode":"json","api_key":NCBI_API_KEY},
        timeout=30
    ).json()
    ids = resp.get("esearchresult", {}).get("idlist", [])
    if not ids:
        return ""
    all_text = ""
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        all_text += requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params={"db":"pubmed","id":",".join(batch),"rettype":"abstract","retmode":"text","api_key":NCBI_API_KEY},
            timeout=30
        ).text
        time.sleep(0.5)
    return all_text

def clean_abstract(text):
    for kw in ["Declarations","Conflict of interest","Ethical Approval","Competing interests","©","Funding","Disclosures"]:
        m = re.compile(re.escape(kw), re.IGNORECASE).search(text)
        if m: text = text[:m.start()].strip()
    return text

def parse_abstracts(raw):
    papers = []
    for entry in re.split(r'\n\d+\.', raw):
        if "Author information:" not in entry: continue
        pm = re.search(r'PMID:\s*(\d+)', entry)
        if not pm: continue
        pmid = pm.group(1)
        parts = entry.split("Author information:")[-1].split("\n\n")
        abs_parts = [p.strip() for p in parts[1:] if len(p.strip()) > 100]
        abstract = clean_abstract(" ".join(abs_parts)) if abs_parts else None
        if abstract and len(abstract) > 50:
            papers.append({"abstract": abstract, "pmid": pmid,
                           "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"})
    return papers

def fetch_metadata(pmids):
    data = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
        params={"db":"pubmed","id":",".join(pmids),"retmode":"json","api_key":NCBI_API_KEY},
        timeout=30
    ).json()
    out = {}
    for pmid, rec in data.get("result", {}).items():
        if pmid == "uids" or not isinstance(rec, dict): continue
        out[pmid] = {
            "title": rec.get("title", "Unknown"),
            "authors": ", ".join(a["name"] for a in rec.get("authors", []) if isinstance(a, dict)) or "Unknown",
            "year": rec.get("pubdate", "")[:4],
        }
    return out

In [ ]:
CORPUS_TOPIC = "Stem cell therapy Parkinson's disease"

raw = fetch_pubmed_abstracts_batched(CORPUS_TOPIC, max_results=100)
papers = parse_abstracts(raw)
meta = fetch_metadata([p["pmid"] for p in papers])
for p in papers:
    if p["pmid"] in meta:
        p.update(meta[p["pmid"]])

print(f"Corpus: {len(papers)} abstracts fetched and parsed")

# Save corpus so it's fixed across all encoder runs
with open("eval_corpus.json", "w") as f:
    json.dump(papers, f)
print("Corpus saved to eval_corpus.json")

## 3. Encoders

Three encoders are compared. All use the same ChromaDB retrieval logic — the only variable is the embedding model.

In [ ]:
ENCODERS = {
    "MiniLM":      "all-MiniLM-L6-v2",
    "PubMedBERT":  "microsoft/BiomedNLP-PubMedBERT-base-uncased",
    "BioLORD":     "FremyCompany/BioLORD-2023",
}

## 4. Retrieval

For each encoder, embed the corpus, store in ChromaDB, then retrieve top 5 abstracts for each query.

In [ ]:
def build_collection(papers, model, collection_name):
    client = chromadb.Client()
    try: client.delete_collection(collection_name)
    except: pass
    col = client.create_collection(collection_name)
    embeddings = model.encode([p["abstract"] for p in papers], show_progress_bar=False)
    col.add(
        documents=[p["abstract"] for p in papers],
        embeddings=[e.tolist() for e in embeddings],
        metadatas=[{"title": p.get("title",""), "pmid": p["pmid"],
                    "pubmed_url": p["pubmed_url"]} for p in papers],
        ids=[f"doc_{i}" for i in range(len(papers))]
    )
    return col

def retrieve(query, collection, model, n=5):
    emb = model.encode([query])[0].tolist()
    res = collection.query(query_embeddings=[emb], n_results=n,
                           include=["documents","metadatas"])
    return list(zip(res["documents"][0], res["metadatas"][0]))

def run_retrieval_for_encoder(encoder_name, model_name, papers, queries):
    print(f"\nRunning: {encoder_name} ({model_name})")
    model = SentenceTransformer(model_name)
    col = build_collection(papers, model, f"eval_{encoder_name}")
    results = {}
    for qid, query in queries.items():
        results[qid] = retrieve(query, col, model, n=5)
    print(f"  Retrieved top-5 for {len(results)} queries")
    return results

In [ ]:
# Load corpus
with open("eval_corpus.json") as f:
    papers = json.load(f)

# Run all three encoders
all_results = {}
for name, model_name in ENCODERS.items():
    all_results[name] = run_retrieval_for_encoder(name, model_name, papers, QUERIES)

print("\nRetrieval complete for all encoders.")

## 5. Build Pooled Annotation Set

For each query, collect the union of all abstracts retrieved by any encoder. 
You annotate each unique (query, abstract) pair once — this avoids judging the same abstract multiple times.

In [ ]:
# Build pool: for each query, unique PMIDs retrieved by any encoder
pool = {}  # pool[qid] = {pmid: {title, abstract, pubmed_url}}

for qid in QUERIES:
    pool[qid] = {}
    for encoder_name, results in all_results.items():
        for doc, meta in results[qid]:
            pmid = meta["pmid"]
            if pmid not in pool[qid]:
                pool[qid][pmid] = {
                    "title": meta["title"],
                    "pmid": pmid,
                    "pubmed_url": meta["pubmed_url"],
                    "abstract_preview": doc[:300]
                }

total_pairs = sum(len(v) for v in pool.values())
print(f"Total unique (query, abstract) pairs to annotate: {total_pairs}")
print(f"(vs {len(QUERIES) * 5 * len(ENCODERS)} if annotating each encoder separately)\n")

for qid, abstracts in pool.items():
    print(f"{qid}: {len(abstracts)} unique abstracts to judge")

## 6. Manual Annotation

For each query, review each abstract in the pool and assign:
- **1** = relevant (directly addresses the query)
- **0** = not relevant (tangential or unrelated)

Run this cell to display abstracts one query at a time and enter your judgements.

In [ ]:
import sys

def annotate_pool(pool, queries, save_path="annotations.json"):
    # Load existing annotations if resuming
    try:
        with open(save_path) as f:
            annotations = json.load(f)
        print(f"Resuming from existing annotations ({sum(len(v) for v in annotations.values())} pairs already done)")
    except:
        annotations = {}

    for qid, abstracts in pool.items():
        if qid not in annotations:
            annotations[qid] = {}

        pending = {pmid: info for pmid, info in abstracts.items()
                   if pmid not in annotations[qid]}
        if not pending:
            print(f"{qid}: already annotated, skipping")
            continue

        print(f"\n{'='*70}")
        print(f"{qid}: {queries[qid]}")
        print(f"{len(pending)} abstracts to judge")
        print('='*70)

        for pmid, info in pending.items():
            print(f"\nTitle: {info['title']}")
            print(f"URL:   {info['pubmed_url']}")
            print(f"Preview: {info['abstract_preview']}...")
            while True:
                label = input("Relevant? [1=yes / 0=no / s=skip query]: ").strip()
                if label in ("0", "1"):
                    annotations[qid][pmid] = int(label)
                    break
                elif label == "s":
                    print(f"Skipping remainder of {qid}")
                    break
                else:
                    print("Enter 1, 0, or s")
            if label == "s":
                break

        # Save after each query in case of interruption
        with open(save_path, "w") as f:
            json.dump(annotations, f, indent=2)
        print(f"  Saved annotations for {qid}")

    print(f"\nAnnotation complete. Saved to {save_path}")
    return annotations

# Uncomment to run annotation session:
# annotations = annotate_pool(pool, QUERIES)

> **To annotate:** uncomment the last line and run the cell. You can stop and resume at any time — progress is saved after each query to `annotations.json`.

## 7. Compute Precision@5

Once annotation is complete, compute Precision@5 for each encoder and query.

In [ ]:
def precision_at_k(retrieved_pmids, annotations_for_query, k=5):
    hits = sum(annotations_for_query.get(pmid, 0) for pmid in retrieved_pmids[:k])
    return hits / k

def score_encoder(encoder_name, results, annotations, queries, k=5):
    scores = {}
    for qid in queries:
        if qid not in annotations:
            continue
        retrieved_pmids = [meta["pmid"] for _, meta in results[qid]]
        scores[qid] = precision_at_k(retrieved_pmids, annotations[qid], k)
    return scores

# Load annotations
with open("annotations.json") as f:
    annotations = json.load(f)

# Score all encoders
all_scores = {}
for encoder_name, results in all_results.items():
    all_scores[encoder_name] = score_encoder(encoder_name, results, annotations, QUERIES)

# Build results table
in_scope = [q for q in QUERIES if not q.startswith("Q12") and
            not q.startswith("Q13") and not q.startswith("Q14")]
out_scope = ["Q12", "Q13", "Q14"]

rows = []
for encoder_name, scores in all_scores.items():
    rows.append({
        "Encoder": encoder_name,
        "P@5 (in-scope, mean)":    round(np.mean([scores.get(q,0) for q in in_scope]), 3),
        "P@5 (out-of-scope, mean)": round(np.mean([scores.get(q,0) for q in out_scope]), 3),
        "P@5 (overall mean)":       round(np.mean(list(scores.values())), 3),
    })

df = pd.DataFrame(rows).set_index("Encoder")
print(df.to_string())

## 8. Per-query breakdown

In [ ]:
# Per-query scores for each encoder
rows2 = []
for qid, query in QUERIES.items():
    row = {"Query ID": qid, "Query": query[:60]+"..."}
    for encoder_name, scores in all_scores.items():
        row[encoder_name] = scores.get(qid, "—")
    rows2.append(row)

df2 = pd.DataFrame(rows2).set_index("Query ID")
print(df2.to_string())

## 9. Interpretation

Write your interpretation here after reviewing the scores.

**Template:**
- Which encoder performed best overall, and by how much?
- Were there specific query types where domain-specific encoders showed the clearest advantage?
- Did any encoder retrieve more false positives on out-of-scope queries?
- Does the result support or contradict the hypothesis that domain-specific pretraining improves biomedical retrieval?
- What are the limitations of this evaluation (corpus size, single annotator, abstract-only text)?